In [ ]:
# Load cleaned data, define financial features, and establish the training cutoff to prevent temporal data leakage.
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 50)
df = pd.read_parquet("../data/interim_clean.parquet")
CUT = pd.Timestamp("2014-12-31")
fit = df[df.issue_d <= CUT].copy()
amt = ["annual_inc", "loan_amnt", "installment", "revol_bal", "tot_cur_bal", "avg_cur_bal", "total_rev_hi_lim", "tot_hi_cred_lim"]
ratio = ["dti", "revol_util", "bc_util"]
print(len(df), len(fit), fit.issue_d.max())

In [ ]:
# full  stastical distribution profile,for the financial amount and ratio features
# Profile financial features to assess distributions, missing values, skewness, and potential outliers.
def profile(s):
    q = s.quantile([.01, .05, .25, .5, .75, .95, .99])
    return pd.Series({"missing": s.isna().sum(), "zeros": (s == 0).sum(), "negatives": (s < 0).sum(),
        "mean": s.mean(), "median": s.median(), "std": s.std(), "IQR": q[.75] - q[.25], "skew": s.skew(),
        "p1": q[.01], "p5": q[.05], "p25": q[.25], "p75": q[.75], "p95": q[.95], "p99": q[.99], "max": s.max()})
prof = pd.DataFrame({c: profile(fit[c]) for c in amt + ratio}).T
prof.round(2)

In [ ]:
# Compare 3-standard-deviation, IQR, and robust Z-score methods to quantify potential outliers.
# compare three outlier rules
cols = amt + ratio
def rules(s):
    s = s.dropna(); m, sd = s.mean(), s.std()
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    med = s.median(); mad = (s - med).abs().median(); rz = 0.6745 * (s - med) / mad
    return pd.Series({"3sd": ((s < m - 3*sd) | (s > m + 3*sd)).sum(),
                      "iqr": ((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum(), "robust_z": (rz.abs() > 3.5).sum()})
out = pd.DataFrame({c: rules(fit[c]) for c in cols}).T
n = fit[cols].notna().sum()
(out.div(n, axis=0) * 100).round(2).assign(n=n)

In [ ]:
# Validate negative utilization values and investigate extreme annual incomes.
calc = fit.revol_bal / fit.total_rev_hi_lim.replace(0, np.nan) * 100
neg, ok = fit.revol_util < 0, fit.revol_util >= 0
print("negative revol_util:", neg.sum(), "| negative bc_util:", (fit.bc_util < 0).sum(), "| both:", (neg & (fit.bc_util < 0)).sum())
print(fit.revol_util[neg].describe().round(1))
print("median gap vs recomputed | negatives (abs):", (fit.revol_util[neg].abs() - calc[neg]).abs().median().round(2), "| normal rows:", (fit.revol_util[ok] - calc[ok]).abs().median().round(2))
print(fit.nlargest(8, "annual_inc")[["annual_inc", "loan_amnt", "dti", "verification_status", "emp_title", "defaulted"]])

In [ ]:
outlog = pd.DataFrame({c: rules(np.log1p(fit[c])) for c in amt}).T
(outlog.div(fit[amt].notna().sum(), axis=0) * 100).round(2)

In [ ]:
 # Correct negative utilization values, cap extreme financial values, apply log transformation, and evaluate treatment effects.
import json
for c in ["revol_util", "bc_util"]:
    df.loc[df[c] < 0, c] = df[c].abs()
fit = df[df.issue_d <= CUT].copy()
heavy = ["annual_inc", "revol_bal", "tot_cur_bal", "avg_cur_bal", "total_rev_hi_lim", "tot_hi_cred_lim"]
spec = {c: {"cap_p99.5": float(fit[c].quantile(.995))} for c in heavy + ["revol_util", "bc_util"]}
json.dump(spec, open("../reports/caps.json", "w"), indent=1)

rows = []
for c in heavy:
    x, cap = fit[c], spec[c]["cap_p99.5"]; t = np.log1p(x.clip(upper=cap)); hi = x > cap
    rows.append(dict(field=c, cap=round(cap), n_capped=hi.sum(), skew_raw=x.skew(), skew_treated=t.skew(),
        sd_over_iqr_raw=x.std() / (x.quantile(.75) - x.quantile(.25)), sd_over_iqr_treated=t.std() / (t.quantile(.75) - t.quantile(.25)),
        default_capped=fit.defaulted[hi].mean(), default_rest=fit.defaulted[~hi].mean()))
pd.DataFrame(rows).set_index("field").round(2)

In [ ]:
fig, ax = plt.subplots(len(heavy), 2, figsize=(10, 14))
for i, c in enumerate(heavy):
    ax[i, 0].hist(fit[c], bins=60); ax[i, 0].set_title(f"{c} (linear)")
    ax[i, 1].hist(np.log1p(fit[c]), bins=60); ax[i, 1].set_title(f"log1p({c})")
plt.tight_layout(); plt.savefig("../reports/financial_distributions.png", dpi=110); plt.show()

In [ ]:
# Compare borrowers' financial profiles by mortgage-account ownership. financial  value 
g = fit.groupby(fit.mort_acc > 0)
print(g.size().div(len(fit)).round(3).rename("share"))
g[["tot_cur_bal", "avg_cur_bal", "tot_hi_cred_lim"]].median().round(0)

In [ ]:
# Compare default rates between borrowers with and without mortgage accounts.
print(fit.groupby(fit.mort_acc > 0).defaulted.agg(["mean", "count"]).round(3))

In [ ]:
# Investigate missing and zero credit-utilization values and compare their default rates.
m = fit.revol_util.isna()
print("revol_util missing:", m.sum(), "| of which total_rev_hi_lim == 0:", (m & (fit.total_rev_hi_lim == 0)).sum(), "| all zero-limit loans:", (fit.total_rev_hi_lim == 0).sum())
z = fit.revol_bal == 0
print("zero revol_bal:", z.sum(), "| of which revol_util == 0:", (z & (fit.revol_util == 0)).sum())
print(fit.groupby(fit.bc_util.isna()).defaulted.agg(["mean", "count"]).round(3))
print(fit.groupby(fit.revol_bal == 0).defaulted.agg(["mean", "count"]).round(3))

In [ ]:
# Define reproducible capping and missing-value treatments, then validate thresholds on the 2015 period.
spec["dti"] = {"cap_p99.5": float(fit.dti.quantile(.995))}
spec["missing"] = {
    "bc_util": {"strategy": "fill with fit median + flag bc_util_missing", "fill": float(fit.bc_util.median())},
    "revol_util": {"strategy": "fill with fit median, no flag (4 rows)", "fill": float(fit.revol_util.median())}}
json.dump(spec, open("../reports/caps.json", "w"), indent=1)
late = df[df.issue_d > CUT]
chk = {c: (late[c] > v["cap_p99.5"]).mean() for c, v in spec.items() if "cap_p99.5" in v}
print(pd.Series(chk).round(4))

In [ ]:
# Validate the training-period DTI cap against 2015 data and assess default risk among borrowers exceeding the threshold.
print(pd.DataFrame({"fit": fit.dti, "2015": late.dti}).describe(percentiles=[.5, .9, .99, .995]).round(2))
hi = late.dti > spec["dti"]["cap_p99.5"]
print("cap:", round(spec["dti"]["cap_p99.5"], 2), "| 2015 above cap:", hi.sum(), "| default rate above/below:", late.defaulted[hi].mean().round(3), late.defaulted[~hi].mean().round(3))

In [ ]:
# . Update the DTI configuration
spec["dti"] = {"cap_p99.5": None, "valid_range": [0, 100],
               "reason": "no percentile cap: 2015 loans above the old p99.5 (36.94) default at 34.5% vs 19.8% below; open-ended top WoE bin instead"}
json.dump(spec, open("../reports/caps.json", "w"), indent=1)
spec["dti"]

In [ ]:
# Assess default-risk patterns across financial feature deciles before and after preprocessing.
t = {}
for c in heavy:
    trt = np.log1p(fit[c].clip(upper=spec[c]["cap_p99.5"]))
    for name, x in [("raw", fit[c]), ("treated", trt)]:
        t[(c, name)] = fit.groupby(pd.qcut(x.rank(method="first"), 10, labels=False)).defaulted.mean()
pd.DataFrame(t).round(3)

In [ ]:
# persist the cleaned data, there is not negetaive utilization left 
df.to_parquet("../data/clean_v2.parquet", index=False)
print(len(df), df.revol_util.min(), df.bc_util.min())

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
df = pd.read_parquet("../data/clean_v2.parquet")
fit = df[df.issue_d <= pd.Timestamp("2014-12-31")]
rest = ["loan_amnt", "installment", "dti", "revol_util", "bc_util"]
fig, ax = plt.subplots(len(rest), 2, figsize=(10, 12))
for i, c in enumerate(rest):
    ax[i, 0].hist(fit[c].dropna(), bins=60); ax[i, 0].set_title(f"{c} (linear)")
    ax[i, 1].hist(np.log1p(fit[c].dropna()), bins=60); ax[i, 1].set_title(f"log1p({c})")
plt.tight_layout(); plt.savefig("../reports/financial_distributions_2.png", dpi=110); plt.show()